# 매출수량 예측하기

### 1. Import & Seed

In [50]:
import os
import random
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from tqdm import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

### 2. Data Loading and Feature Engineering

In [ ]:
# Load data
train_df = pd.read_csv('./train/train.csv')
train_df['영업일자'] = pd.to_datetime(train_df['영업일자'])

# 날짜 파싱 & 기본 캘린더 피처
train_df["dow"] = train_df["영업일자"].dt.dayofweek       # 요일 (0=월 ~ 6=일)
train_df["is_weekend"] = (train_df["dow"] >= 5).astype(int)
train_df["month"] = train_df["영업일자"].dt.month
train_df["quarter"] = train_df["영업일자"].dt.quarter
train_df["doy"] = train_df["영업일자"].dt.dayofyear

# Label Encode menu names and create a venue name column
le = LabelEncoder()
train_df['영업장명_메뉴명'] = train_df['영업장명_메뉴명'].astype(str)
train_df['영업장명_메뉴명_encoded'] = le.fit_transform(train_df['영업장명_메뉴명'])
train_df['업장명'] = train_df['영업장명_메뉴명'].apply(lambda x: x.split('_')[0])

# Sort data to ensure correct time series order
train_df = train_df.sort_values(by=['영업장명_메뉴명_encoded', '영업일자']).reset_index(drop=True)

# Feature Engineering Function for date components
def make_features(df):
    df_features = df.copy()
    df_features['dayofweek'] = df_features['영업일자'].dt.dayofweek
    df_features['month'] = df_features['영업일자'].dt.month
    df_features['day'] = df_features['영업일자'].dt.day
    df_features['weekofyear'] = df_features['영업일자'].dt.isocalendar().week.astype(int)
    return df_features

print("Creating features for the entire training dataset...")
train_featured = make_features(train_df)
train_featured.fillna(0, inplace=True)

Creating features for the entire training dataset...


### 3. Model Training (Single-Step Prediction)

In [52]:
# Filter out zero-sales days from training data
train_final = train_featured[train_featured['매출수량'] > 0].copy()

# --- Feature Engineering with Lags and Rolling Windows ---
def create_lag_rolling_features(df):
    df_copy = df.copy()
    df_copy = df_copy.sort_values(by=['영업장명_메뉴명_encoded', '영업일자'])
    grouped = df_copy.groupby('영업장명_메뉴명_encoded')['매출수량']
    
    for lag in [7, 14, 21, 28]:
        df_copy[f'lag_{lag}'] = grouped.shift(lag)
        
    for window in [7, 14, 28]:
        series_shifted = grouped.shift(7) # Use data from at least a week ago
        df_copy[f'rolling_mean_{window}'] = series_shifted.rolling(window).mean()
        df_copy[f'rolling_std_{window}'] = series_shifted.rolling(window).std()
        
    df_copy.fillna(0, inplace=True)
    return df_copy

print("Creating lag and rolling features for training data...")
train_with_features = create_lag_rolling_features(train_final)
print("Feature creation complete.")

# --- Model Training for Single-Step Prediction ---
features = [col for col in train_with_features.columns if col not in ['영업일자', '영업장명_메뉴명', '매출수량', '업장명']]

# Target is the next day's sales (t+1)
y_train = train_with_features.groupby('영업장명_메뉴명_encoded')['매출수량'].shift(-1)

X_train = train_with_features[features]

# Align X and y by dropping rows with NaN targets (the last day of each series)
valid_indices = y_train.dropna().index
X_train = X_train.loc[valid_indices]
y_train = y_train.loc[valid_indices]

# Create sample weights
train_subset_for_weights = train_final.loc[X_train.index]
sample_weights = np.ones(len(X_train))
important_indices = train_subset_for_weights[train_subset_for_weights['업장명'].isin(['담하', '미라시아'])].index
aligned_indices = X_train.index.intersection(important_indices)
sample_weights[X_train.index.get_indexer(aligned_indices)] = 2.0

# Train the final model (no MultiOutputRegressor)
print(f"Training final model on {len(X_train)} samples...")
final_model = lgb.LGBMRegressor(
    objective="tweedie",        # Tweedie 분포 회귀
    tweedie_variance_power=1.3, # 1<p<2, 보통 1.1~1.5 범위에서 튜닝
    learning_rate=0.05,
    n_estimators=2000,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
final_model.fit(X_train, y_train, sample_weight=sample_weights)
print("Final model training complete.")

Creating lag and rolling features for training data...
Feature creation complete.
Training final model on 48435 samples...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2848
[LightGBM] [Info] Number of data points in the train set: 48435, number of used features: 15
[LightGBM] [Info] Start training from score 2.955911
Final model training complete.


### 4. Prediction and Submission File Generation (Autoregressive)

In [53]:
# Load the submission template
submission_df = pd.read_csv('./sample_submission.csv')

test_files = sorted(glob.glob('./test/TEST_*.csv'))

for test_file in tqdm(test_files, desc="Generating predictions"):
    # 1. Load the 28-day input data and add all features
    history = pd.read_csv(test_file)
    history['영업일자'] = pd.to_datetime(history['영업일자'])
    history = make_features(history)  # Add date features to the initial history
    history['영업장명_메뉴명'] = history['영업장명_메뉴명'].astype(str)
    history['영업장명_메뉴명_encoded'] = le.transform(history['영업장명_메뉴명'])
    history['업장명'] = history['영업장명_메뉴명'].apply(lambda x: x.split('_')[0])
    test_prefix = os.path.basename(test_file).split('.')[0]

    # 2. Autoregressively predict for the next 7 days
    for i in range(1, 8):
        last_known_date = history['영업일자'].max()
        pred_date = last_known_date + pd.Timedelta(days=1)
        
        # 3. Create a dataframe for the prediction date
        pred_df = pd.DataFrame({'영업일자': [pred_date] * len(le.classes_), '영업장명_메뉴명': le.classes_})
        pred_df = make_features(pred_df)  # Add date features
        pred_df['영업장명_메뉴명_encoded'] = le.transform(pred_df['영업장명_메뉴명'])
        pred_df['업장명'] = pred_df['영업장명_메뉴명'].apply(lambda x: x.split('_')[0])

        # 4. Create lag/rolling features from history
        # Concat full history and the new prediction rows
        temp_history = pd.concat([history, pred_df], ignore_index=True)
        
        # Use ffill to carry over the last known sales value for the prediction date row
        temp_history['매출수량'] = temp_history.groupby('영업장명_메뉴명_encoded')['매출수량'].ffill().fillna(0)
        
        # Create lag/rolling features on the combined frame
        temp_with_features = create_lag_rolling_features(temp_history)
        
        # Select the feature set for the specific date we want to predict
        pred_features_df = temp_with_features[temp_with_features['영업일자'] == pred_date].copy()
        
        # 5. Predict and update history
        predictions = final_model.predict(pred_features_df[features])
        pred_df['매출수량'] = np.maximum(0, predictions)
        
        # Append the full prediction (with features) to the history for the next day's forecast
        history = pd.concat([history, pred_df], ignore_index=True)

        # 6. Directly overwrite the main submission dataframe
        date_str = test_prefix + '+' + str(i) + '일'
        row_idx = submission_df[submission_df['영업일자'] == date_str].index
        if not row_idx.empty:
            day_pred_dict = pred_df.set_index('영업장명_메뉴명')['매출수량'].to_dict()
            for col in submission_df.columns[1:]:
                submission_df.loc[row_idx, col] = day_pred_dict.get(col, 0)

# 7. Final processing: fill any NaNs that might have occurred and save
submission_df.fillna(0, inplace=True)
submission_df.to_csv('lgbm_submission.csv', index=False, encoding='utf-8-sig')

print("Final submission file created: lgbm_submission.csv")

Generating predictions:   0%|          | 0/10 [00:00<?, ?it/s]/var/folders/3l/wd0qm1dn76x6yl12pgj5l90c0000gn/T/ipykernel_35389/1509798142.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.7229460208504066' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission_df.loc[row_idx, col] = day_pred_dict.get(col, 0)
/var/folders/3l/wd0qm1dn76x6yl12pgj5l90c0000gn/T/ipykernel_35389/1509798142.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.816851612724832' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission_df.loc[row_idx, col] = day_pred_dict.get(col, 0)
/var/folders/3l/wd0qm1dn76x6yl12pgj5l90c0000gn/T/ipykernel_35389/1509798142.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in 

Final submission file created: lgbm_submission.csv
